In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow import keras  # pyright: ignore[reportMissingModuleSource]
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
import keras_tuner as kt  # pyright: ignore[reportMissingModuleSource]

keras.utils.set_random_seed(42)
t00 = time.time()

In [ ]:
def calc_performance(y_true, y_pred, labels):
    """Confusion matrix and classification report for a multi-class problem.

    Same helper as Module 3 and the in-class Keras notebook.
    y_true and y_pred are integer labels, not one-hot vectors.
    """
    confusion_matrix_data = confusion_matrix(y_true, y_pred, labels=labels)
    confusion_matrix_display = ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix_data, display_labels=labels
    )
    report_text = classification_report(y_true, y_pred, digits=4)
    report_dict = classification_report(y_true, y_pred, output_dict=True, digits=4)
    return {
        "confusion_matrix_data": confusion_matrix_data,
        "confusion_matrix_display": confusion_matrix_display,
        "report_text": report_text,
        "report_dict": report_dict,
    }

def plot_train_val(history_df, title_prefix=""):
    """Plot training and validation accuracy and loss against epoch.

    history_df must have columns epoch, accuracy, val_accuracy, loss, val_loss.
    Returns the two Axes, (ax_acc, ax_loss).
    """
    fig, ax_acc = plt.subplots(figsize=(8, 4))
    ax_acc.plot(history_df['epoch'], history_df['accuracy'], marker='o', color='blue', label='train')
    ax_acc.plot(history_df['epoch'], history_df['val_accuracy'], marker='o', color='red', label='validation')
    ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_title(title_prefix + 'Accuracy vs Epoch')
    ax_acc.legend()
    ax_acc.grid(True)
    fig.tight_layout()
    plt.show()

    fig, ax_loss = plt.subplots(figsize=(8, 4))
    ax_loss.plot(history_df['epoch'], history_df['loss'], marker='o', color='blue', label='train')
    ax_loss.plot(history_df['epoch'], history_df['val_loss'], marker='o', color='red', label='validation')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_title(title_prefix + 'Loss vs Epoch')
    ax_loss.legend()
    ax_loss.grid(True)
    fig.tight_layout()
    plt.show()
    return ax_acc, ax_loss

In [ ]:
dfHold = pd.read_csv('C:\\Users\\blake\\Downloads\\archive\\sudoku.csv')
df_sudoku_full = dfHold.iloc[:500000, :]
df_sudoku_full = df_sudoku_full.astype({"puzzle": str, "solution": str})
df_sudoku_full['puzzle'] = df_sudoku_full['puzzle'].str.zfill(81)

In [ ]:
print(dfHold.head())
print(df_sudoku_full.head())

                                              puzzle  \
0  7000004304000961080063490009405200035846002000...   
1  3010865040465210705000000014008000020803479000...   
2  4830156036000809091067000302000093550901020067...   
3  8317000004205109000040070327160904901450000045...   
4  4089063000013682080074051900046705245002070026...   

                                            solution  
0  6795182435437296188216349577943521863584617292...  
1  3719865248465213795924738614638197522853479167...  
2  7483915623652487919126754834217869355894132766...  
3  2983176457642851391539462783271689549814537266...  
4  1428956379751368248367425193984671524513287962...  
                                              puzzle  \
0  0700000430400096108006349000940520003584600200...   
1  3010865040465210705000000014008000020803479000...   
2  0483015603600080909106700030200009355090102006...   
3  0083170000042051090000400703271609049014500000...   
4  0408906300001368208007405190004670524500207002... 

In [ ]:
puzzles_array = (np.array([list(map(int, list(puzzle))) for puzzle in df_sudoku_full['puzzle']]))
solutions_array = (np.array([list(map(int, list(solution))) for solution in df_sudoku_full['solution']]))
print(puzzles_array.shape)
print(puzzles_array[0])
print(solutions_array.shape)
print(solutions_array[0])

(500000, 81)
[0 7 0 0 0 0 0 4 3 0 4 0 0 0 9 6 1 0 8 0 0 6 3 4 9 0 0 0 9 4 0 5 2 0 0 0 3
 5 8 4 6 0 0 2 0 0 0 0 8 0 0 5 3 0 0 8 0 0 7 0 0 9 1 9 0 2 1 0 0 0 0 5 0 0
 7 0 4 0 8 0 2]
(500000, 81)
[6 7 9 5 1 8 2 4 3 5 4 3 7 2 9 6 1 8 8 2 1 6 3 4 9 5 7 7 9 4 3 5 2 1 8 6 3
 5 8 4 6 1 7 2 9 2 1 6 8 9 7 5 3 4 4 8 5 2 7 6 3 9 1 9 6 2 1 8 3 4 7 5 1 3
 7 9 4 5 8 6 2]


In [ ]:
one_hot_puzzles = np.zeros((puzzles_array.shape[0], 9, 9, 10), dtype=np.float32)
for i in range(puzzles_array.shape[0]):
    for j in range(9):
        for k in range(9):
            value_puz = puzzles_array[i, j * 9 + k]
            one_hot_puzzles[i, j, k, value_puz] = 1.0
print(one_hot_puzzles.shape)
print(one_hot_puzzles[0])

(500000, 9, 9, 10)
[[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
  [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]]

 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
  [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0.

In [ ]:
train_puzzles = one_hot_puzzles[:80000]
val_puzzles = one_hot_puzzles[80000:90000]
test_puzzles = one_hot_puzzles[90000:]

In [9]:
def build_model(n_layers, filters, learning_rate):
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(9, 9, 10)))
    for n in range(n_layers):
        model.add(keras.layers.Conv2D(filters, (3, 3), padding='same'))
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Activation('relu'))
    model.add(keras.layers.Conv2D(9, (1, 1), activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_model(n_layers=15, filters=512, learning_rate=1e-3)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 9, 9, 512)      │        46,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 33,119,241 (126.34 MB)

 Trainable params: 33,103,881 (126.28 MB)

 Non-trainable params: 15,360 (60.00 KB)

In [ ]:
train_solutions_int = solutions_array[:80000].reshape(-1, 9, 9) - 1
val_solutions_int = solutions_array[80000:90000].reshape(-1, 9, 9) - 1
test_solutions_int = solutions_array[90000:].reshape(-1, 9, 9) - 1

In [ ]:
callbacks_final = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

model.fit(
    train_puzzles, train_solutions_int,
    validation_data=(val_puzzles, val_solutions_int),
    epochs=50,
    batch_size=32,
    callbacks=callbacks_final
)

Epoch 1/50
  23/2500 ━━━━━━━━━━━━━━━━━━━━ 35:42 865ms/step - accuracy: 0.1116 - loss: 2.3967

In [ ]:
results = calc_performance()

In [11]:
def solve_puzzle(model, puzzle):
    grid = puzzle.copy()
    while (grid[:, :, 0] == 1).any():
        probs = model.predict(grid[np.newaxis], verbose=0)[0]
        best_conf = -1
        best_j = 0
        best_k = 0
        for j in range(9):
            for k in range(9):
                if grid[j, k, 0] == 1 and probs[j, k].max() > best_conf:
                    best_conf = probs[j, k].max()
                    best_j = j
                    best_k = k
        digit = probs[best_j, best_k].argmax() + 1
        grid[best_j, best_k, :] = 0
        grid[best_j, best_k, digit] = 1.0
    return grid.argmax(axis=-1)

solved = solve_puzzle(model, test_puzzles[0])
print(solved)
print((solved == solutions_array[90000].reshape(9, 9)).all())

NameError: name 'model' is not defined

In [ ]:
model.save('sudoku_solver_model.keras')

In [15]:
model = keras.models.load_model('sudoku_solver_model_large.keras')

for i, puz in enumerate(test_puzzles):
    correct = 0
    solved = solve_puzzle(model, puz)
    if solved.all() == test_solutions_int[i].all():
        correct += 1
    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1} puzzles, correct so far: {correct}")

accuracy = correct / len(test_puzzles)
print(f"Test accuracy: {accuracy:.4f}")

KeyboardInterrupt: 